# Furniture Object Detection with YOLOv8 

We will train YOLO to detect **Chair, Sofa and Table** using the supplied ZIP file.

This notebook still performs the complete workflow:

```text
ZIP → prepare dataset → train → evaluate → predict → save model
```

Place `Furniture.v1-furnitureobjects.yolov8.zip` in the same folder as this notebook. In Google Colab, upload it to `/content` first.

File contents:
| Content        |  Images | Label files |
| -------------- | ------: | ----------: |
| Training set   |     544 |         544 |
| Testing set    |     146 |         146 |
| Validation set | Missing |     Missing |
Folder structure:
Furniture.v1-furnitureobjects.yolov8/
│
├── data.yaml
├── README.dataset.txt
├── README.roboflow.txt
│
├── train/
│   ├── images/
│   └── labels/
│
└── test/
    ├── images/
    └── labels/
train/images ------------

Contains 544 furniture photographs used to teach YOLO.

train/labels---------------

Contains one annotation .txt file for every training image. These files tell YOLO:

Which object is present
Where the object is located
The width and height of its bounding box
test/images

Contains 146 images used to evaluate the trained model on unseen data.

test/labels

Contains the correct bounding-box annotations for the test images.

data.yaml

This configuration file tells YOLO where the dataset is located and which objects it contains:

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 3
names: ['Chair', 'Sofa', 'Table']

Here:

nc: 3 means there are three object classes.
Class 0 = Chair
Class 1 = Sofa
Class 2 = Table
One label in the dataset contains:

0 0.45708 0.425 0.47417 0.81

YOLO label format is:

class_id  x_center  y_center  width  height

## 1. Install YOLO

In [2]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


- `%pip install` installs a Python package in the notebook environment.

- `ultralytics` provides the YOLO class.

**Inference:** No red error means installation was successful.

## 2. Import libraries and check the hardware

In [3]:
from pathlib import Path
import random, shutil
import matplotlib.pyplot as plt
import torch, yaml
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


- `Path` handles file paths.
- `random` selects validation and test images.
- `shutil` moves/copies files.
- `shutil` also extracts the ZIP.
- `matplotlib` displays predictions.
- `torch` checks GPU availability.
- `yaml` creates YOLO's dataset configuration.
- `YOLO` loads, trains and uses the model.
- `DEVICE=0` means the first GPU; `"cpu"` means the processor.

**Inference:** A GPU makes training much faster, but CPU also works.

## 3. Extract the dataset

In [10]:
DATASET = Path("Furniture.v1-furnitureobjects.yolov8")

print("Dataset folder:", DATASET.resolve())

Dataset folder: C:\Users\lenovo\Downloads\Furniture.v1-furnitureobjects.yolov8


- `ZIP_FILE` is the exact ZIP filename in the notebook folder.
- `DATASET` is the folder created when that ZIP is extracted.
- `.exists()` prevents extraction from repeating.
- `unpack_archive(ZIP_FILE, ".")` extracts the ZIP into the current notebook folder.

**Inference:** The printed path is the dataset's main folder.

## 4. Inspect the classes and dataset size

In [11]:
print("Classes: Chair, Sofa, Table")
for split in ["train", "valid", "test"]:
    images = len(list((DATASET/split/"images").glob("*.jpg")))
    labels = len(list((DATASET/split/"labels").glob("*.txt")))
    print(split, "images:", images, "labels:", labels)

Classes: Chair, Sofa, Table
train images: 0 labels: 0
valid images: 0 labels: 0
test images: 0 labels: 0


- `glob("*.jpg")` finds JPG images; `glob("*.txt")` finds labels.
- `len()` counts the files found.
- Each image should have a corresponding label file.

**Expected inference:** The dataset contains classes `Chair`, `Sofa`, and `Table`. It has training and test data but originally has no validation data.

## 5. Create the missing validation set and corrected YAML

In [12]:
random.seed(42)
train_img, train_lbl = DATASET/"train"/"images", DATASET/"train"/"labels"
valid_img, valid_lbl = DATASET/"valid"/"images", DATASET/"valid"/"labels"
valid_img.mkdir(parents=True, exist_ok=True)
valid_lbl.mkdir(parents=True, exist_ok=True)

if not list(valid_img.glob("*.jpg")):
    all_images = list(train_img.glob("*.jpg"))
    chosen = random.sample(all_images, round(len(all_images) * 0.20))
    for image in chosen:
        shutil.move(image, valid_img/image.name)
        shutil.move(train_lbl/f"{image.stem}.txt", valid_lbl/f"{image.stem}.txt")

DATA_YAML = Path("furniture_data.yaml")
config = {"path": str(DATASET.resolve()), "train": "train/images",
          "val": "valid/images", "test": "test/images",
          "names": {0: "Chair", 1: "Sofa", 2: "Table"}}
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))

print("Train images:", len(list(train_img.glob("*.jpg"))))
print("Validation images:", len(list(valid_img.glob("*.jpg"))))
print(DATA_YAML.read_text())

Train images: 0
Validation images: 0
path: C:\Users\lenovo\Downloads\Furniture.v1-furnitureobjects.yolov8
train: train/images
val: valid/images
test: test/images
names:
  0: Chair
  1: Sofa
  2: Table



### Lines and parameters

- `random.seed(42)` makes the same random choice reproducible.
- `mkdir(..., exist_ok=True)` creates validation folders safely.
- `0.20` reserves 20% of training images for validation.
- `random.sample()` selects unique images.
- `image.stem` is the filename without `.jpg`; the matching label uses the same stem.
- `shutil.move()` creates a **non-overlapping** validation set by removing selected pairs from training.
- The `if` condition prevents the split from being repeated when the cell is rerun.
- `path`, `train`, `val`, and `test` tell YOLO where the data is.
- `names` maps IDs: `0=Chair`, `1=Sofa`, `2=Table`.

**Inference:** Approximately 80% remains for training and 20% becomes validation. Test images remain untouched.

## 6. Load the small pretrained YOLOv8 model

In [13]:
model = YOLO("yolov8n.pt")
print("Model loaded")

Model loaded


- `yolov8n` means YOLOv8 **Nano**, the smallest and fastest YOLOv8 model.
- `.pt` is a PyTorch weights file.
- It downloads automatically the first time.
- Starting with pretrained weights is called **transfer learning**.

## 7. Train the model

In [15]:
model.train(
    data=str(DATA_YAML),
    epochs=2,
    imgsz=640,
    batch=8,
    device=DEVICE,
    patience=5,
    project="runs",
    name="furniture",
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.120 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.118  Python-3.13.9 torch-2.13.0+cpu CPU (AMD Ryzen 5 4600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=furniture_data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tra

FileNotFoundError: [34m[1mtrain: [0mError loading data from C:\Users\lenovo\Downloads\Furniture.v1-furnitureobjects.yolov8\train\images
See https://docs.ultralytics.com/datasets for dataset formatting guidance.

### Training parameters

| Parameter | Meaning |
|---|---|
| `data` | Path to our dataset YAML. |
| `epochs=20` | Twenty complete passes through the training data. Use 5 for a quick demonstration. |
| `imgsz=640` | Resizes input to 640 × 640 pixels. |
| `batch=8` | Processes eight images before updating weights. Reduce it if memory runs out. |
| `device` | Uses the selected GPU or CPU. |
| `workers=0` | Avoids common multiprocessing errors in Windows Jupyter. |
| `patience=5` | Stops if validation does not improve for five epochs. |
| `project`, `name` | Saves results inside `runs/furniture`. |
| `exist_ok=True` | Allows reuse of this output-folder name. |

**Inference:** Falling losses and rising validation mAP normally indicate learning. Exact values vary each run.

## 8. Load the best model and evaluate it

In [16]:
from pathlib import Path
from ultralytics import YOLO

best_files = list(Path(".").rglob("best.pt"))

if len(best_files) == 0:
    print("best.pt was not found.")
    print("Run the training cell completely before running this cell.")

else:
    BEST_PATH = best_files[-1]

    print("Loading model from:")
    print(BEST_PATH.resolve())

    best_model = YOLO(str(BEST_PATH))

    metrics = best_model.val(
        data=str(DATA_YAML),
        device=DEVICE,
    
    )

    print(f"Precision: {metrics.box.mp:.3f}")
    print(f"Recall: {metrics.box.mr:.3f}")
    print(f"mAP50: {metrics.box.map50:.3f}")
    print(f"mAP50-95: {metrics.box.map:.3f}")

best.pt was not found.
Run the training cell completely before running this cell.


- `best.pt` contains weights from the best validation epoch.
- `.val()` evaluates the detector on validation images.
- **Precision:** How many predicted objects were correct?
- **Recall:** How many actual objects were found?
- **mAP50:** Detection quality using an IoU threshold of 0.50.
- **mAP50–95:** A stricter average over IoU thresholds from 0.50 to 0.95.
- Values are usually between 0 and 1; higher is generally better.

## 9. Predict furniture in unseen test images

In [ ]:
test_images = list((DATASET/"test"/"images").glob("*.jpg"))
samples = random.sample(test_images, min(4, len(test_images)))

results = best_model.predict(
    source=[str(x) for x in samples],
    conf=0.25,
    imgsz=640,
    save=True
)

- `random.sample()` chooses up to four unseen test images.
- `source` supplies their paths to YOLO.
- `conf=0.25` keeps predictions having at least 25% confidence.
- `imgsz=640` uses the same image size as training.
- `save=True` saves images with boxes and labels.

**Inference:** Increasing `conf` gives fewer, more confident detections; reducing it finds more candidates but may add false detections.

## 10. Display predictions

In [ ]:
%matplotlib inline
plt.figure(figsize=(12, 5 * len(results)))

for i, result in enumerate(results, 1):
    image = result.plot()[..., ::-1]
    plt.subplot(len(results), 1, i)
    plt.imshow(image)
    plt.title(Path(result.path).name)
    plt.axis("off")

plt.tight_layout()
plt.show()

- `result.plot()` draws bounding boxes, class names and confidence scores.
- `[..., ::-1]` converts the plotted image from BGR to RGB for Matplotlib.
- `enumerate(..., 1)` provides image numbers starting from 1.
- `subplot()` places all results in one figure.
- `axis("off")` hides graph axes.

**Inference:** `Sofa 0.86`, for example, means YOLO is 86% confident that the box contains a sofa.

## 11. Save the trained model with a clear name

In [ ]:
FINAL_MODEL = Path("furniture_yolov8n_best.pt")
shutil.copy2(BEST_PATH, FINAL_MODEL)
print("Saved model:", FINAL_MODEL.resolve())

- `copy2()` copies `best.pt` to an easy-to-recognize filename.
- This `.pt` file is the final trained detector.

## Final inference

The dataset supplies labelled examples. YOLOv8 learns from them and saves the learned weights in `furniture_yolov8n_best.pt`. The trained model can then detect Chairs, Sofas and Tables in new images.